# Check a Qwen drafter

Loads `drafter.pt` and checks it on CodeSearchNet functions the training notebook's bench split did not print as the sample. Nothing here updates weights.

The output has to equal greedy Qwen. A faster run that changes a token is a failure.


In [ ]:
%pip install -q "transformers>=5.3,<6" "datasets>=3" "huggingface-hub>=1.32,<2" accelerate sentencepiece protobuf


In [ ]:
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

CANDIDATES = [
    Path("/content/qwen-draft/drafter.pt"),
    Path("/content/drive/MyDrive/qwen-draft/drafter.pt"),
    Path("drafter.pt"),
]
N_TEST = 32
NEW_TOKENS = 64
SEED = 99

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then rerun.")
device = torch.device("cuda")
path = next((item for item in CANDIDATES if item.exists()), None)
if path is None:
    try:
        from google.colab import files
        print("No drafter.pt found. Upload the file saved by the training notebook.")
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("No file uploaded.")
        name = next(iter(uploaded))
        path = Path("/content") / name
    except ImportError:
        raise SystemExit("Put drafter.pt at /content/qwen-draft/drafter.pt and rerun this cell.")

ckpt = torch.load(path, map_location="cpu", weights_only=False)
needed = ["model", "vocab", "context", "prefix", "propose", "draft_dim", "draft_layers", "draft_heads", "drafter"]
missing = [key for key in needed if key not in ckpt]
if missing:
    raise SystemExit(f"{path} is missing {missing}.")
print(f"loaded {path}")
print(f"teacher {ckpt['model']} | propose {ckpt['propose']} | trained tokens/forward {ckpt.get('tokens_per_qwen_forward', float('nan')):.2f}")


In [ ]:
class Drafter(nn.Module):
    def __init__(self, vocab: int, dim: int, n_layer: int, n_head: int, context: int):
        super().__init__()
        self.context = context
        self.embed = nn.Embedding(vocab, dim)
        self.pos = nn.Embedding(context, dim)
        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    dim, n_head, dim * 4, dropout=0.0, batch_first=True, norm_first=True, activation="gelu"
                )
                for _ in range(n_layer)
            ]
        )
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.embed.weight

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        length = tokens.shape[1]
        if length > self.context:
            tokens = tokens[:, -self.context :]
            length = tokens.shape[1]
        index = torch.arange(length, device=tokens.device)
        x = self.embed(tokens) + self.pos(index)
        mask = torch.triu(torch.full((length, length), float("-inf"), device=tokens.device), diagonal=1)
        for block in self.blocks:
            x = block(x, src_mask=mask)
        return self.head(self.norm(x))


model_id = ckpt["model"]
tokenizer = AutoTokenizer.from_pretrained(model_id)
context = int(ckpt["context"])
prefix_len = int(ckpt["prefix"])
propose = int(ckpt["propose"])
drafter = Drafter(
    int(ckpt["vocab"]), int(ckpt["draft_dim"]), int(ckpt["draft_layers"]), int(ckpt["draft_heads"]), context
).to(device)
drafter.load_state_dict(ckpt["drafter"])
drafter.eval()
teacher = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, low_cpu_mem_usage=True)
teacher.to(device)
teacher.eval()


def windows(limit: int) -> torch.Tensor:
    ds = load_dataset(ckpt.get("dataset", "code-search-net/code_search_net"), ckpt.get("dataset_config", "python"), split="test", streaming=True)
    ds = ds.shuffle(buffer_size=2048, seed=SEED)
    ids: list[int] = []
    need = limit * context
    for row in ds:
        text = (row.get("whole_func_string") or "").replace("\r\n", "\n").strip()
        if len(text) < 40 or "def " not in text:
            continue
        piece = tokenizer(text + "\n", add_special_tokens=False)["input_ids"]
        if len(piece) < 16 or len(piece) > 1024:
            continue
        ids.extend(piece)
        if len(ids) >= need + context:
            break
    usable = (len(ids) // context) * context
    if usable < need:
        raise SystemExit(f"test split produced {usable // context} windows, need {limit}.")
    data = torch.tensor(ids[:usable], dtype=torch.long).view(-1, context)
    order = torch.randperm(data.shape[0], generator=torch.Generator().manual_seed(SEED))
    return data[order[:limit]]


held = windows(N_TEST)
print(tokenizer.decode(held[0].tolist(), skip_special_tokens=True)[:400])


In [ ]:
@torch.no_grad()
def greedy(prefix: torch.Tensor, n_new: int) -> torch.Tensor:
    out = prefix.view(1, -1)
    for _ in range(n_new):
        nxt = teacher(out).logits[:, -1].argmax(-1, keepdim=True)
        out = torch.cat([out, nxt], dim=1)
    return out[0]


@torch.no_grad()
def speculate(prefix: torch.Tensor, n_new: int, k: int) -> tuple[torch.Tensor, int]:
    out = prefix.view(1, -1)
    start = out.shape[1]
    forwards = 0
    while out.shape[1] - start < n_new:
        take = min(k, n_new - (out.shape[1] - start))
        draft = out
        proposed = []
        for _ in range(take):
            nxt = drafter(draft).logits[:, -1].argmax(-1, keepdim=True)
            proposed.append(nxt)
            draft = torch.cat([draft, nxt], dim=1)
        props = torch.cat(proposed, dim=1)
        full = torch.cat([out, props], dim=1)
        logits = teacher(full).logits[0]
        base = out.shape[1] - 1
        forwards += 1
        n_keep = 0
        for i in range(take):
            if int(logits[base + i].argmax()) == int(props[0, i]):
                n_keep += 1
            else:
                break
        if n_keep:
            out = torch.cat([out, props[:, :n_keep]], dim=1)
        if out.shape[1] - start >= n_new:
            break
        if n_keep < take:
            out = torch.cat([out, logits[base + n_keep].argmax().view(1, 1)], dim=1)
        else:
            out = torch.cat([out, logits[base + take].argmax().view(1, 1)], dim=1)
    return out[0, : start + n_new], forwards


same = 0
forwards = 0
plain_ms = 0.0
spec_ms = 0.0
show = None
for i in tqdm(range(held.shape[0]), desc="check", mininterval=1, dynamic_ncols=True):
    prefix = held[i, :prefix_len].to(device)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    gold = greedy(prefix, NEW_TOKENS)
    torch.cuda.synchronize()
    plain_ms += (time.perf_counter() - t0) * 1000
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    got, n_forward = speculate(prefix, NEW_TOKENS, propose)
    torch.cuda.synchronize()
    spec_ms += (time.perf_counter() - t1) * 1000
    match = torch.equal(gold, got)
    same += int(match)
    forwards += n_forward
    if show is None:
        show = (gold, got, match)

n = held.shape[0]
per_forward = (n * NEW_TOKENS) / max(1, forwards)
print(f"matched greedy Qwen on {same}/{n}")
print(f"Qwen forwards per continuation: greedy {NEW_TOKENS}, speculative {forwards / n:.1f}")
print(f"tokens per Qwen forward: {per_forward:.2f}")
print(f"ms per continuation: greedy {plain_ms / n:.0f}, speculative {spec_ms / n:.0f}")
if same != n:
    raise SystemExit("Speculative text differed from greedy Qwen.")
if per_forward <= 1.05:
    print("No speedup on this split.")
else:
    print("Fewer Qwen forwards. The text is still Qwen's.")
gold, got, match = show
print("\n--- greedy Qwen ---")
print(tokenizer.decode(gold.tolist(), skip_special_tokens=True)[:700])
print("\n--- speculative ---")
print(tokenizer.decode(got.tolist(), skip_special_tokens=True)[:700])
